# Cross-Defect Contamination Check

**Goal:** Test whether images labeled as *no defect* for one defect type  
actually contain a *different* defect — using the fine-tuned binary models.

| Data folder | Tested on | Question |
|-------------|-----------|----------|
| `data/No_Warping` | Cracking model + Stringing model | Are these images cracked or stringed? |
| `data/No_Cracking` | Warping model + Stringing model | Are these images warped or stringed? |
| `data/No_Stringing` | Warping model + Cracking model | Are these images warped or cracked? |

Each binary model outputs a probability in [0, 1].  
Threshold = **0.5** → prediction = defect present if prob ≥ 0.5.

> Preprocessing is identical to training: resize 224×224, normalize to [0, 1].

---
## 1. Imports & Shared Utilities

In [2]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow import keras

BASE  = os.getcwd()           # Defect-Detection/
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
THRESHOLD  = 0.5
AUTOTUNE   = tf.data.AUTOTUNE

# ── Shared helpers ────────────────────────────────────────────────────────────

def make_dataset(folder):
    """Load all .jpg images from folder into a tf.data pipeline (no labels)."""
    paths = sorted(glob.glob(os.path.join(folder, "*.jpg")))
    print(f"  Images found: {len(paths)}")

    def _load(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32) / 255.0
        return img

    ds = (
        tf.data.Dataset.from_tensor_slices(paths)
        .map(_load, num_parallel_calls=AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )
    return ds, len(paths)


def run_inference(model_path, ds, n_images, defect_label, no_defect_label):
    """Load model, predict, and print a short report."""
    model = keras.models.load_model(model_path)
    probs = model.predict(ds, verbose=0).ravel()

    n_defect    = (probs >= THRESHOLD).sum()
    n_no_defect = (probs <  THRESHOLD).sum()
    pct         = n_defect / n_images * 100

    print(f"  Model       : {os.path.basename(model_path)}")
    print(f"  Predicted {defect_label:<14}: {n_defect:>5}  ({pct:.1f}%)")
    print(f"  Predicted {no_defect_label:<14}: {n_no_defect:>5}  ({100-pct:.1f}%)")
    print(f"  Mean prob   : {probs.mean():.4f}  |  Max: {probs.max():.4f}")
    return probs


print("✅ Helpers ready.")

2026-05-14 21:37:17.665051: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778791037.781243    5002 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778791037.812164    5002 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-14 21:37:18.075014: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ Helpers ready.


---
## 2. `No_Warping` → Cracking model + Stringing model

These 1,815 images are labeled as **no warping**.  
We check whether any of them are flagged as **cracked** or **stringed**.

In [3]:
NO_WARPING_DIR     = os.path.join(BASE, "data", "No_Warping")
CRACKING_MODEL     = os.path.join(BASE, "cracking",  "mobilenetv2_cracking_finetuned_final.h5")
STRINGING_MODEL    = os.path.join(BASE, "Stringing", "mobilenetv2_stringing_finetuned_final.h5")

print("=" * 55)
print("Data : No_Warping")
print("=" * 55)
ds_nw, n_nw = make_dataset(NO_WARPING_DIR)

print()
print("--- Cracking model ---")
probs_nw_crack = run_inference(CRACKING_MODEL, ds_nw, n_nw, "Cracking", "No_Cracking")

print()
print("--- Stringing model ---")
probs_nw_str = run_inference(STRINGING_MODEL, ds_nw, n_nw, "Stringing", "No_Stringing")

Data : No_Warping
  Images found: 1815


I0000 00:00:1778791043.816780    5002 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6100 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060 SUPER, pci bus id: 0000:01:00.0, compute capability: 7.5



--- Cracking model ---


I0000 00:00:1778791049.462543    5146 service.cc:148] XLA service 0x7ed2640031e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778791049.463297    5146 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 2060 SUPER, Compute Capability 7.5
2026-05-14 21:37:29.522657: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778791049.924028    5146 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1778791051.188821    5146 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Model       : mobilenetv2_cracking_finetuned_final.h5
  Predicted Cracking      :    77  (4.2%)
  Predicted No_Cracking   :  1738  (95.8%)
  Mean prob   : 0.1558  |  Max: 0.8131

--- Stringing model ---


  Model       : mobilenetv2_stringing_finetuned_final.h5
  Predicted Stringing     :  1555  (85.7%)
  Predicted No_Stringing  :   260  (14.3%)
  Mean prob   : 0.8253  |  Max: 1.0000


---
## 3. `No_Cracking` → Warping model + Stringing model

These 2,042 images are labeled as **no cracking**.  
We check whether any of them are flagged as **warped** or **stringed**.

In [4]:
NO_CRACKING_DIR    = os.path.join(BASE, "data", "No_Cracking")
WARPING_MODEL      = os.path.join(BASE, "warping",   "mobilenetv2_Warping_finetuned_final.h5")

print("=" * 55)
print("Data : No_Cracking")
print("=" * 55)
ds_nc, n_nc = make_dataset(NO_CRACKING_DIR)

print()
print("--- Warping model ---")
probs_nc_warp = run_inference(WARPING_MODEL, ds_nc, n_nc, "Warping", "No_Warping")

print()
print("--- Stringing model ---")
probs_nc_str = run_inference(STRINGING_MODEL, ds_nc, n_nc, "Stringing", "No_Stringing")

Data : No_Cracking
  Images found: 2042

--- Warping model ---


  Model       : mobilenetv2_Warping_finetuned_final.h5
  Predicted Warping       :     1  (0.0%)
  Predicted No_Warping    :  2041  (100.0%)
  Mean prob   : 0.0217  |  Max: 0.5917

--- Stringing model ---


  Model       : mobilenetv2_stringing_finetuned_final.h5
  Predicted Stringing     :  2033  (99.6%)
  Predicted No_Stringing  :     9  (0.4%)
  Mean prob   : 0.9847  |  Max: 1.0000


---
## 4. `No_Stringing` → Warping model + Cracking model

These 1,811 images are labeled as **no stringing**.  
We check whether any of them are flagged as **warped** or **cracked**.

In [5]:
NO_STRINGING_DIR   = os.path.join(BASE, "data", "No_Stringing")

print("=" * 55)
print("Data : No_Stringing")
print("=" * 55)
ds_ns, n_ns = make_dataset(NO_STRINGING_DIR)

print()
print("--- Warping model ---")
probs_ns_warp = run_inference(WARPING_MODEL, ds_ns, n_ns, "Warping", "No_Warping")

print()
print("--- Cracking model ---")
probs_ns_crack = run_inference(CRACKING_MODEL, ds_ns, n_ns, "Cracking", "No_Cracking")

Data : No_Stringing
  Images found: 1811

--- Warping model ---


  Model       : mobilenetv2_Warping_finetuned_final.h5
  Predicted Warping       :     0  (0.0%)
  Predicted No_Warping    :  1811  (100.0%)
  Mean prob   : 0.0169  |  Max: 0.4910

--- Cracking model ---


  Model       : mobilenetv2_cracking_finetuned_final.h5
  Predicted Cracking      :    32  (1.8%)
  Predicted No_Cracking   :  1779  (98.2%)
  Mean prob   : 0.0359  |  Max: 0.8680


---
## 5. Summary Table

Collect all results in one table for easy reading.

In [6]:
results = [
    ("No_Warping",  n_nw,  "Cracking",  probs_nw_crack),
    ("No_Warping",  n_nw,  "Stringing", probs_nw_str),
    ("No_Cracking", n_nc,  "Warping",   probs_nc_warp),
    ("No_Cracking", n_nc,  "Stringing", probs_nc_str),
    ("No_Stringing",n_ns,  "Warping",   probs_ns_warp),
    ("No_Stringing",n_ns,  "Cracking",  probs_ns_crack),
]

print(f"{'Data folder':<15} {'Model':<12} {'Total':>6} {'Flagged':>8} {'%':>6}  {'Mean prob':>10}")
print("-" * 62)
for folder, n, defect, probs in results:
    flagged = int((probs >= THRESHOLD).sum())
    pct     = flagged / n * 100
    print(f"{folder:<15} {defect:<12} {n:>6} {flagged:>8} {pct:>5.1f}%  {probs.mean():>10.4f}")

Data folder     Model         Total  Flagged      %   Mean prob
--------------------------------------------------------------
No_Warping      Cracking       1815       77   4.2%      0.1558
No_Warping      Stringing      1815     1555  85.7%      0.8253
No_Cracking     Warping        2042        1   0.0%      0.0217
No_Cracking     Stringing      2042     2033  99.6%      0.9847
No_Stringing    Warping        1811        0   0.0%      0.0169
No_Stringing    Cracking       1811       32   1.8%      0.0359
